# Pipeline RAG — Notebook Kaggle reproductible

## Question-answering sur la documentation technique

Ce notebook constitue le protocole expérimental principal du projet. Il exécute les étapes dans un ordre déterministe, conserve les artefacts dans un répertoire persistant, réutilise les modèles déjà chargés et distingue explicitement les scores valides des erreurs d'évaluation.

Le corpus couvre la documentation officielle de **Python**, **Scikit-learn** et **LangChain**. Les métriques de qualité sont rapportées séparément pour la recherche documentaire et la génération. Aucun score global arbitraire n'est utilisé pour éviter de masquer les échecs d'une composante.

> **Pré-requis Kaggle :** activer Internet et un GPU T4. Le téléchargement du corpus et du modèle peut prendre plusieurs minutes.


## 0. Configuration expérimentale

Les chemins sont séparés entre le code source et les données produites. Cette organisation évite qu'un nouveau clonage supprime le corpus, l'index FAISS ou les résultats. Le notebook peut donc être relancé par étapes.


In [ ]:
# Installation ciblée : Kaggle fournit déjà généralement torch et transformers.
%pip install -q gitpython ftfy tiktoken sentence-transformers faiss-cpu accelerate bitsandbytes gradio


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import random
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

REPO_URL = "https://github.com/DAHANIElkhalil25/rag-pipeline.git"
WORK_DIR = Path("/kaggle/working/rag-pipeline")
DATA_DIR = Path("/kaggle/working/rag_data")
RESULTS_DIR = DATA_DIR / "evaluation"

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK_DIR)], check=True)
os.environ["RAG_DATA_DIR"] = str(DATA_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

# Reload project modules from the freshly cloned source.
for module_name in ["config", "etape5_generation", "etape6_evaluation", "ui"]:
    sys.modules.pop(module_name, None)

from config import (
    BENCHMARK_DIR,
    CLEAN_DIR,
    EVALUATION_DIR,
    LLM_CONFIG,
    METADATA_DIR,
    VECTORSTORE_DIR,
    init_directories,
)
init_directories()

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(f"Répertoire de travail : {WORK_DIR}")
print(f"Répertoire des données : {DATA_DIR}")
print(f"Commit source : {commit}")
print(f"GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


## 1. Collecte documentaire

Les documents sont collectés depuis les dépôts officiels. La collecte est exécutée seulement si le corpus n'existe pas déjà. Les versions et le commit source sont conservés dans le manifeste expérimental.


In [ ]:
from etape1_collecte import main as run_collection

raw_index = METADATA_DIR / "corpus_index.csv"
if not raw_index.exists():
    run_collection()
else:
    print(f"Corpus existant réutilisé : {raw_index}")


## 2. Nettoyage et normalisation

Les opérations de nettoyage sont appliquées avant le chunking afin de réduire le bruit structurel, de préserver les blocs de code et d'éviter d'indexer des doublons.


In [ ]:
from etape2_nettoyage import main as run_cleaning

clean_index = CLEAN_DIR / "corpus_cleaned_index.csv"
if not clean_index.exists():
    run_cleaning()
else:
    print(f"Corpus nettoyé existant réutilisé : {clean_index}")


## 3. Benchmarking du retrieval

Le benchmarking compare les configurations de chunking, d'embedding et de recherche sur un jeu de questions annoté par mots-clés. Les métriques sont **Hit Rate@k**, **MRR@k** et **Precision@k**. Le rapport JSON généré est la seule source de configuration pour l'index final.

Cette étape peut durer longtemps sur Kaggle. Elle n'est exécutée que si `benchmark_report.json` est absent.


In [ ]:
from etape3_benchmarking import main as run_benchmarking

benchmark_report = BENCHMARK_DIR / "benchmark_report.json"
if not benchmark_report.exists():
    run_benchmarking()
else:
    print(f"Rapport de benchmark réutilisé : {benchmark_report}")


In [ ]:
with open(benchmark_report, "r", encoding="utf-8") as handle:
    benchmark = json.load(handle)
print(json.dumps(benchmark.get("recommendations", {}), ensure_ascii=False, indent=2))


## 4. Indexation FAISS

Les embeddings sont normalisés et indexés avec `IndexFlatIP`, de sorte que le produit scalaire corresponde à la similarité cosinus. L'index et les métadonnées sont conservés ensemble.


In [ ]:
from etape4_indexation import main as run_indexing

faiss_index = VECTORSTORE_DIR / "faiss_index.bin"
chunks_metadata = VECTORSTORE_DIR / "chunks_metadata.json"
if not (faiss_index.exists() and chunks_metadata.exists()):
    run_indexing()
else:
    print(f"Index existant réutilisé : {faiss_index}")


## 5. Chargement unique du pipeline

Le pipeline est chargé **une seule fois**. Cette décision est essentielle sur une carte T4 : le notebook ne doit pas créer une seconde instance de Mistral pendant l'évaluation, car cela provoquait les erreurs CUDA Out of Memory observées dans l'ancienne version.


In [ ]:
from etape5_generation import load_pipeline

pipeline = load_pipeline()
print(f"Chunks indexés : {len(pipeline.chunks)}")
print(f"Configuration de recherche : {pipeline.search_config}")


## 6. Vérification qualitative

Avant l'évaluation quantitative, quelques questions de contrôle permettent de vérifier la cohérence des réponses, la présence des sources et le comportement multilingue du système. Les réponses complètes sont sauvegardées en JSON et CSV.


In [ ]:
demo_questions = [
    ("Python", "What is a Python decorator and how is it used?"),
    ("Scikit-learn", "What is the difference between fit() and fit_transform()?"),
    ("LangChain", "What is the role of a retriever in LangChain?"),
]

demo_results = []
for domain, question in demo_questions:
    started = time.time()
    result = pipeline.answer(question)
    demo_results.append({
        "domain": domain,
        "question": question,
        "answer": result.get("answer", ""),
        "sources": result.get("sources", []),
        "retrieval_scores": result.get("retrieval_scores", []),
        "elapsed_seconds": round(time.time() - started, 3),
    })
    print(f"[{domain}] {question}\n{result.get('answer', '')[:800]}\n")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(Path(RESULTS_DIR) / "demo_results.json").write_text(
    json.dumps(demo_results, ensure_ascii=False, indent=2), encoding="utf-8"
)
pd.DataFrame([
    {
        "domain": row["domain"],
        "question": row["question"],
        "answer": row["answer"],
        "n_sources": len(row["sources"]),
        "elapsed_seconds": row["elapsed_seconds"],
    }
    for row in demo_results
]).to_csv(Path(RESULTS_DIR) / "demo_results.csv", index=False)


## 7. Évaluation quantitative

L'évaluateur est volontairement exécuté sur l'instance déjà chargée. Les scores invalides restent absents (`null` dans le JSON et cellule vide dans le CSV) ; ils ne sont pas remplacés par `0.5` et ne sont pas confondus avec un score nul.

Les quatre métriques sont rapportées séparément : **faithfulness**, **answer relevancy**, **context precision** et **context recall**. Les résultats incluent également le nombre de valeurs valides, la couverture, l'écart-type et les erreurs de pipeline.


In [ ]:
from etape6_evaluation import GROUND_TRUTH_QA, run_evaluation

evaluation_report = run_evaluation(
    pipeline=pipeline,
    qa_pairs=GROUND_TRUTH_QA,
    output_dir=EVALUATION_DIR,
)
print(json.dumps(evaluation_report.get("metric_summary", {}), ensure_ascii=False, indent=2))


In [ ]:
report_path = EVALUATION_DIR / "ragas_report.json"
details_path = EVALUATION_DIR / "ragas_details.csv"

report = json.loads(report_path.read_text(encoding="utf-8"))
summary_df = pd.DataFrame(report["metric_summary"]).T.reset_index(names="metric")
details_df = pd.read_csv(details_path)

summary_df.to_csv(EVALUATION_DIR / "evaluation_summary.csv", index=False)
details_df.to_csv(EVALUATION_DIR / "evaluation_details_normalized.csv", index=False)

print("Résumé :")
display(summary_df)
print("\nRépartition des statuts :")
display(details_df["Status"].value_counts(dropna=False).rename_axis("status").to_frame("count"))


## 8. Visualisation des résultats

Les graphiques décrivent les moyennes par domaine et la couverture des métriques. Les valeurs absentes sont ignorées pour les moyennes, mais leur nombre est affiché séparément.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
metric_cols = ["Faithfulness", "Answer_Relevancy", "Context_Precision", "Context_Recall"]
for col in metric_cols:
    details_df[col] = pd.to_numeric(details_df[col], errors="coerce")

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = details_df.groupby("Source", dropna=False)[metric_cols].mean().T
plot_df.plot(kind="bar", ax=ax)
ax.set_ylim(0, 1)
ax.set_ylabel("Score moyen valide")
ax.set_xlabel("Métrique")
ax.set_title("Qualité moyenne par source documentaire")
ax.legend(title="Source")
fig.tight_layout()
fig.savefig(EVALUATION_DIR / "evaluation_by_source.png", dpi=160)
plt.show()

coverage = summary_df[["metric", "n_valid", "n_total", "coverage"]].copy()
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=coverage, x="metric", y="coverage", color="#4472C4", ax=ax)
ax.set_ylim(0, 1)
ax.set_ylabel("Couverture des scores valides")
ax.set_xlabel("")
ax.set_title("Fiabilité opérationnelle de l'évaluation")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(EVALUATION_DIR / "evaluation_coverage.png", dpi=160)
plt.show()


## 9. Interface utilisateur

L'interface est lancée en dernière étape, après le chargement du pipeline. Elle réutilise exactement la même instance et expose les sources récupérées avec leur rang. Sur Kaggle, `share=True` génère un lien public temporaire affiché dans la sortie de la cellule.


In [ ]:
from config import UI_CONFIG
from ui import launch_ui

# Mettre à False pour terminer le notebook sans ouvrir le serveur.
LAUNCH_UI = True
if LAUNCH_UI:
    demo = launch_ui(
        pipeline,
        share=UI_CONFIG["share"],
        server_name=UI_CONFIG["server_name"],
        server_port=UI_CONFIG["server_port"],
    )


## 10. Export des artefacts

Le dossier d'évaluation contient les réponses, les scores détaillés, le résumé statistique, les métadonnées de validité et les graphiques. Il peut être téléchargé depuis l'onglet **Files** de Kaggle.


In [ ]:
manifest = {
    "repository": REPO_URL,
    "commit": commit,
    "seed": SEED,
    "data_dir": str(DATA_DIR),
    "llm_config": LLM_CONFIG,
    "benchmark_report": str(benchmark_report),
    "evaluation_report": str(report_path),
    "files": sorted(str(path.relative_to(DATA_DIR)) for path in DATA_DIR.rglob("*") if path.is_file()),
}
(DATA_DIR / "run_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

archive = Path("/kaggle/working/rag_results_bundle")
if archive.exists():
    shutil.rmtree(archive)
shutil.copytree(DATA_DIR, archive)
shutil.make_archive("/kaggle/working/rag_results_bundle", "zip", archive)
print("Archive créée : /kaggle/working/rag_results_bundle.zip")


## 11. Références méthodologiques

Les choix méthodologiques s'appuient sur les références suivantes :

1. Lewis, P. et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. [NeurIPS / arXiv](https://arxiv.org/abs/2005.11401).
2. Reimers, N. & Gurevych, I. (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks*. [EMNLP](https://aclanthology.org/D19-1410/).
3. Johnson, J., Douze, M. & Jégou, H. (2019). *Billion-scale similarity search with GPUs*. [IEEE Transactions on Big Data](https://arxiv.org/abs/1702.08734).
4. Es, S. et al. (2023). *RAGAS: Automated Evaluation of Retrieval Augmented Generation*. [arXiv](https://arxiv.org/abs/2309.15217).
5. Gao, Y. et al. (2023). *Retrieval-Augmented Generation for Large Language Models: A Survey*. [arXiv](https://arxiv.org/abs/2312.10997).

Les scores doivent être interprétés conjointement avec le taux de couverture des métriques et le nombre d'erreurs. Ils ne constituent pas une preuve absolue de la qualité du système ni une probabilité de vérité.
